Make the voxelised electron density

In [4]:
!python 1_voxelize_pdb.py --topology "data/1crn.psf" --trajectory "data/1crn_eq_aligned.dcd" --bbmax 41 --save_path data --system "1crn"

Using MD trajectory mode
Using data/1crn.psf as reference PDB
Using data/1crn_eq_aligned.dcd as trajectory
bbmax: 41.0
n_voxels: 52
cutoff: 5.0
pdb_path: None
trajectory: data/1crn_eq_aligned.dcd
reference_pdb: None
topology: data/1crn.psf
save_path: data
system: 1crn
selection: protein and not element H
creating universe...
/nobackup/proj/disk/naiss2025-23-641/shared/cardoch/xssdense/.venv/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
done.
bounding box: 41.0 Å, num. of voxels: 52, voxel size: 1.61 Å
Computing density for each frame...
done./10000
Saving density to data...
Saving density

Tranform densities to tfrecord for fast training

In [5]:
!python 2_generate_tfrecord.py  --h5 "data/1crn_41bb_1d61dv_5co.h5" --output_dir "data/1crn_41bb_1d61dv_5co.tfrecord"

2026-11-02 11:01:33.390872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-02 11:01:33.399721: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-02 11:01:33.403799: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
h5: data/1crn_41bb_1d61dv_5co.h5
output_dir: data/1crn_41bb_1d61dv_5co.tfrecord
block: 100
test_every: 10
num_proc: 8
Loaded 10000 frames of (52, 52, 52) rho_norm, voxel size 1.6100 Å
Electrons per frame: mean 2205, rel. std 0.00%
Split: 9000 train, 1000 test
Normalization: min 0, max 1.57592
Writing test.tfrecords: 100%|████████████| 1000/1000 [00:00<00:00, 3150

Do the training

In [16]:
!python 3_train_VAE.py  --datapath "data/1crn_41bb_1d61dv_5co.tfrecord" --savepath "data" --epochs=30 --mode constant --beta_input 1e-1 --batchsize 64

GPU DETECTED: TensorFlow found 1 GPU(s):
  GPU 0: /physical_device:GPU:0
datapath: data/1crn_41bb_1d61dv_5co.tfrecord
train_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/tfrecords/train.tfrecords
test_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/tfrecords/test.tfrecords
meta_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
savepath: data
mode: constant
beta_input: 0.1
z_dim: 8
learning_rate: 0.0003
epochs: 30
batchsize: 64
patience: 5
min_improvement: 0.005
save_every_epoch: False
split: 8100 train, 900 validation

Epoch 1/30
Step 0, Loss: 657.51233, MSE: 117.12228, KL: 0.05262
Step 10, Loss: 655.91705, MSE: 116.83708, KL: 0.11455
Step 20, Loss: 653.02710, MSE: 116.33690, KL: 0.88623
Step 30, Loss: 647.22266, MSE: 115.58407, KL: 4.14049
Step 40, Loss: 640.62128, MSE: 126.21791, KL: 18.19224
Step 50, Loss: 459.41971, MSE: 142.28397, KL: 141.64729
Step 60, Loss: 356.20865, MSE: 95.21416, KL: 83.21132
Step 70, Loss: 315.50333, MSE: 77.01093, KL: 26.88340
Step 80, Loss: 303.10229

Compute scattering from voxel density, split into two classes, and save as the reference we want to reconstruct

In [13]:
!python helper/crambin_saxs_class.py --h5 data/1crn_41bb_1d61dv_5co.h5 --out_dir data --rho_bulk 0 --padding 3 --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json

h5: data/1crn_41bb_1d61dv_5co.h5
out_dir: data
padding: 3
n_pca: 2
n_clusters: 2
dark_index: 0
rho_bulk: 0.0
drho: 0.0
shell: 0.0
pa_yield: 1.0
fig_dir: data/figures
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
1000 of 10000 frames are test frames
frame 10000/10000
saved data/1crn_41bb_1d61dv_5co_xss.h5
saved data/1crn_41bb_1d61dv_5co_xss_pca.h5
saved data/figures/curves_norm.png
saved data/figures/pca_clusters.png
saved data/figures/class_intensity.png
saved data/synthetic_class1_vacuum
saved data/synthetic_class2_vacuum
saved data/figures/synthetic_deltaI.png


Run the reconstruction for class 1 and 2

In [17]:
!python 4_genetic_algorithm.py \
    --model_path data/1crn_41bb_1d61dv_5co.vae \
    --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json \
    --h5 data/1crn_41bb_1d61dv_5co.h5 \
    --dI_curve data/synthetic_class1_vacuum/deltaI.csv \
    --dark_curve data/synthetic_class1_vacuum/ground_abs.csv \
    --rho_bulk 0 \
    --target_yield 1 \
    --output_folder results/synthetic_class1_vacuum \
    --padding 3 \
    --qmax 0.6

model_path: data/1crn_41bb_1d61dv_5co.vae
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
h5: data/1crn_41bb_1d61dv_5co.h5
dark_index: 0
dark_curve: data/synthetic_class1_vacuum/ground_abs.csv
dI_curve: data/synthetic_class1_vacuum/deltaI.csv
output_folder: results/synthetic_class1_vacuum
rho_bulk: 0.0
target_yield: 1.0
yield_weight: 2.0
qmax: 0.6
padding: 3
pop_size: 64
generations: 80
n_elite: 5
n_random: 20
mutation_rate: 0.3
top_k: 5
seed: 0
Envelope: 1194 voxels (4983 Å³) for 2205 electrons
Dark fit: scale 1.0000, drho 0.0000, shell 0.0 Å, r2 1.00000
Gen   0  best -1.0661  r2 -1.0661  yield 1.000
Gen   1  best 0.2586  r2 0.2608  yield 0.967
Gen   2  best 0.4230  r2 0.4230  yield 1.000
Gen   3  best 0.4230  r2 0.4230  yield 1.000
Gen   4  best 0.5728  r2 0.5728  yield 1.000
Gen   5  best 0.9609  r2 0.9617  yield 0.980
Gen   6  best 0.9609  r2 0.9617  yield 0.980
Gen   7  best 0.9609  r2 0.9617  yield 0.980
Gen   8  best 0.9609  r2 0.9617  yield 0.980
Gen   9  best 0.9838  r

In [18]:
!python 4_genetic_algorithm.py \
    --model_path data/1crn_41bb_1d61dv_5co.vae \
    --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json \
    --h5 data/1crn_41bb_1d61dv_5co.h5 \
    --dI_curve data/synthetic_class2_vacuum/deltaI.csv \
    --dark_curve data/synthetic_class2_vacuum/ground_abs.csv \
    --rho_bulk 0 \
    --target_yield 1 \
    --output_folder results/synthetic_class2_vacuum \
    --padding 3 \
    --qmax 0.6

model_path: data/1crn_41bb_1d61dv_5co.vae
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
h5: data/1crn_41bb_1d61dv_5co.h5
dark_index: 0
dark_curve: data/synthetic_class2_vacuum/ground_abs.csv
dI_curve: data/synthetic_class2_vacuum/deltaI.csv
output_folder: results/synthetic_class2_vacuum
rho_bulk: 0.0
target_yield: 1.0
yield_weight: 2.0
qmax: 0.6
padding: 3
pop_size: 64
generations: 80
n_elite: 5
n_random: 20
mutation_rate: 0.3
top_k: 5
seed: 0
Envelope: 1194 voxels (4983 Å³) for 2205 electrons
Dark fit: scale 1.0000, drho 0.0000, shell 0.0 Å, r2 1.00000
Gen   0  best -0.0066  r2 -0.0066  yield 1.000
Gen   1  best 0.1521  r2 0.1521  yield 1.000
Gen   2  best 0.1668  r2 0.1668  yield 1.000
Gen   3  best 0.6164  r2 0.6164  yield 1.000
Gen   4  best 0.6163  r2 0.6163  yield 1.000
Gen   5  best 0.7060  r2 0.7060  yield 1.000
Gen   6  best 0.7464  r2 0.7464  yield 1.000
Gen   7  best 0.8998  r2 0.8998  yield 1.000
Gen   8  best 0.8998  r2 0.8998  yield 1.000
Gen   9  best 0.9486  r